# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane and why

Lane 2 — Refresh / Content Opportunity Scoring.
I'm choosing this lane because it's a direct continuation of what I already validated hands-on in Task 1: the starter pipeline (scripts/run_all.py) ranks pages for “refresh review” using exactly this lane's question — which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring? — and I watched a learned model beat a transparent hand rule on that exact task (Precision@50 rose from 0.240 to 0.740, roughly a 3x lift). That's concrete evidence the lane has real signal in it before I've written a single line of my own modeling code, which lowers my risk going into a 7-week project. It also gives me a natural progression: I can start on the small starter CSV this week, then move to the full warehouse's fact_content_daily_performance table from Week 3 onward for a stronger, future-looking label (decline/recovery over a forward window instead of the current-window proxy label the starter pipeline uses).


In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter dataset loaded: {len(df):,} rows, {df.shape[1]} columns")

Starter dataset loaded: 30,000 rows, 44 columns


## 2. The question: decision, action, cost of a wrong call

**The question:** Of all the content pages a client owns, which ones should a human reviewer look at first this week — for a refresh, an expansion, protection, pruning, or just monitoring?

**The decision this improves:** where a content team spends its limited weekly review capacity. Teams can't review every page every week, so the system's job is to rank candidates, not to make the call unilaterally.

**Who acts on it:** a content strategist or SEO editor, using the ranked queue plus its reason codes as a starting point — never as an auto-pilot.

**The action:** the editor opens the top-ranked pages, checks the reason code (e.g. stale_visible_page, declining_with_demand), and decides whether to refresh, expand, protect, prune, or just keep monitoring.

**Cost of a wrong call:**
- False positive (page flagged as worth reviewing, but it wasn't really a priority): wasted reviewer time — low cost, self-correcting.
- False negative (a genuinely declining, high-demand page never surfaces): real cost — the page keeps losing visibility unnoticed. This is the more expensive failure mode.

In [ ]:
false_positive_cost = "wasted reviewer time on a page that wasn't actually urgent"
false_negative_cost = "a real decline goes unnoticed and keeps losing visibility"
print("Optimizing primarily against:", false_negative_cost)
print("Secondary cost to watch:", false_positive_cost)

Optimizing primarily against: a real decline goes unnoticed and keeps losing visibility
Secondary cost to watch: wasted reviewer time on a page that wasn't actually urgent


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter CSV directly and pulled three numbers that argue this lane is worth the next 7 weeks.

In [ ]:
n = len(df)
declining_pct = (df["trend_direction"] == "down").mean()
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()

import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Number 1: Share of the {n:,}-page starter slice currently declining: {declining_pct:.1%}")
print(f"Number 2: Declining pages with real demand (>=100 impressions/90d): {declining_with_demand:,} ({declining_with_demand/n:.1%})")
print(f"Number 3: Precision@50 lift, hand rule vs random forest: {base:.3f} -> {rf:.3f} ({rf/base:.1f}x)")

Number 1: Share of the 30,000-page starter slice currently declining: 54.2%
Number 2: Declining pages with real demand (>=100 impressions/90d): 13,152 (43.8%)
Number 3: Precision@50 lift, hand rule vs random forest: 0.240 -> 0.740 (3.1x)


## 4. Careful words: what I can and can't claim

**What I can claim:** that this lane's ranked output is observed, directional, decision-support — a prioritized starting point for a human reviewer, backed by measurable historical patterns in search/engagement signals. I can say a page's score is elevated because specific, inspectable signals (staleness, demand, position, CTR) point that way.

**What I can never claim:** that refreshing a page causes recovery (that needs an experiment, not this data), that I've reverse-engineered Google's ranking algorithm, or that any single score is a guarantee. I also won't treat FlyRank's own product flags (health_score, is_quick_win, etc.) as ground truth if I ever encounter them — they aren't shipped in this data on purpose.

In [ ]:
print("Lane selected: Refresh / Content Opportunity Scoring")
print("Framing complete. Ready to lock in the ML task type in ML-03 (w02_ml_task_framing.ipynb).")

Lane selected: Refresh / Content Opportunity Scoring
Framing complete. Ready to lock in the ML task type in ML-03 (w02_ml_task_framing.ipynb).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.